# `03_yolo_seg_train.ipynb` — Train YOLO26-seg on ECUSTFD

Trains YOLO26-seg using `src/yolo_seg/train/train_yolo26_seg.py` on the dataset prepared in `02_yolo_seg_pipeline.ipynb`.

## Hyperparameter Policy

Only 2 parameters are overridden from Ultralytics defaults:
- `--imgsz 480` — Hardware constraint: 6.4 GB VRAM OOMs at 640.
- `--batch 4` — Hardware constraint: 6.4 GB VRAM OOMs at batch >= 8 with segmentation head and AMP enabled.

All other parameters follow Ultralytics defaults to avoid ad-hoc tuning. Full hyperparameter configurations are snapshot by Ultralytics to `runs/yolo_seg/<name>/args.yaml`.

## Output Layout

```
runs/yolo_seg/<name>/
├── args.yaml              # Hyperparameter configuration
├── results.csv            # Loss / mAP per epoch
├── results.png            # Training curves
├── weights/{best,last}.pt
models/
└── <name>_{best,last}.pt  # Deployed model checkpoints
```

## Runtime Notes
- **Duration**: ~1–3 hours for 100 epochs on RTX 4050.
- **Resume training**: add `--resume runs/yolo_seg/<name>/weights/last.pt` if interrupted.


## Pre-flight Check

Verifies `ultralytics` installation before running the training script.


In [1]:
# Cell 0 — Pre-flight: đảm bảo ultralytics đã cài
import importlib, subprocess, sys

try:
    spec = importlib.util.find_spec("ultralytics")
    if spec is None:
        raise ImportError("ultralytics not found")
    import ultralytics
    print(f"[OK] ultralytics {ultralytics.__version__} đã có sẵn")
except ImportError:
    print("[..] ultralytics chưa có → đang pip install…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    import ultralytics
    print(f"[OK] vừa cài ultralytics {ultralytics.__version__}")

# Optional: kiểm tra torch + CUDA để biết sẽ train trên CPU hay GPU
try:
    import torch
    cuda_avail = torch.cuda.is_available()
    print(f"[info] torch {torch.__version__}, CUDA available: {cuda_avail}")
    if cuda_avail:
        for i in range(torch.cuda.device_count()):
            name = torch.cuda.get_device_name(i)
            mem  = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"[info]   GPU {i}: {name}  ({mem:.1f} GB)")
except ImportError:
    print("[warn] torch chưa cài (sẽ do ultralytics kéo theo)")

[OK] ultralytics 8.4.87 đã có sẵn
[info] torch 2.13.0+cu130, CUDA available: True
[info]   GPU 0: NVIDIA GeForce RTX 4050 Laptop GPU  (6.0 GB)


## Download Pretrained Weights

Caches pretrained weights to `~/.cache/ultralytics/` prior to training to avoid network timeouts during execution.


In [2]:
# Cell 1 — Download pretrained yolo26n-seg.pt (nếu chưa có)
#
# ultralytics YOLO("yolo26n-seg.pt") sẽ auto-download nếu chưa có trong
# cache ~/.cache/ultralytics/.  Cell này download trước để train không
# bị delay / timeout giữa chừng.
import importlib, subprocess, sys
from pathlib import Path

# 1. Đảm bảo ultralytics đã cài (cell 0 đã làm, nhưng chạy lại cho chắc)
spec = importlib.util.find_spec("ultralytics")
if spec is None:
    print("[install] ultralytics not found → pip install…")
    subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics"],
                   check=True)

from ultralytics import YOLO

# 2. Trigger download (sẽ nhanh nếu đã cached)
MODEL_NAME = "yolo26n-seg.pt"
print(f"[..] Loading YOLO model: {MODEL_NAME}")
model = YOLO(MODEL_NAME)

# 3. Kiểm tra file cached đâu
import os
cache_root = Path.home() / ".cache" / "ultralytics"
cached = list(cache_root.glob("yolo26n-seg.pt")) + list(cache_root.glob("yolo26n*"))
if cached:
    print(f"[OK] Model cached at: {cached[0]}")
else:
    print(f"[OK] Model loaded (downloaded if first run). Cache: {cache_root}")

# 4. In thông tin model
if hasattr(model, "model"):
    n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
    print(f"[info] Model parameters: {n_params:.1f} M")
print(f"[OK] Ready to train.")

[..] Loading YOLO model: yolo26n-seg.pt
[OK] Model loaded (downloaded if first run). Cache: C:\Users\Admin\.cache\ultralytics
[info] Model parameters: 3.1 M
[OK] Ready to train.


In [3]:
# Cell 2 — Train YOLO26-seg
#
# Chạy TRỰC TIẾP trong cell (không qua subprocess) để log Ultralytics
# stream realtime vào cell output của Jupyter — reviewer mở .ipynb sẽ
# thấy đầy đủ history của training (epoch table, GPU mem, ETA). Đồng
# thời vẫn ghi log file riêng vào `data/processed/yolo_ecustfd_seg/logs/`
# để tra cứu offline.
#
# Hyperparameters passed:
#   imgsz=480  batch=4  workers=0  device="0"
# Tất cả hyper khác (epochs=100, lr, optimizer, augmentation, patience,
# close_mosaic, amp, …) để Ultralytics default.
import io
import sys
import shutil
import datetime
from pathlib import Path

# ── Tee so everything printed goes to BOTH cell output and log file ───────
# Cell output gets raw text (including \r for live progress bar overwrites).
# Log file gets clean lines (\r replaced with \n) so reading offline shows
# every progress update as its own line, not just the final one.
class Tee:
    """Split stdout between an original stream (cell output) and a log file.

    - Cell output: raw text (preserves \r for live progress bar overwrites)
    - Log file:    sanitized (\r → \n) so each update is its own line
    """
    def __init__(self, cell_stream, log_fh):
        self.cell_stream = cell_stream
        self.log_fh      = log_fh

    def write(self, s):
        self.cell_stream.write(s)
        if s:
            self.log_fh.write(s.replace("\r", "\n"))
        return len(s)

    def flush(self):
        for t in (self.cell_stream, self.log_fh):
            try:
                t.flush()
            except Exception:
                pass

    def isatty(self):
        # Some libs (rich, Ultralytics console) call isatty() to decide whether
        # to render interactive progress bars. Forward to the underlying stream.
        return getattr(self.cell_stream, "isatty", lambda: False)()

PROJ = Path("E:/AI_Research/dlt8")
LOG_DIR = PROJ / "data" / "processed" / "yolo_ecustfd_seg" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = LOG_DIR / f"train_run_{ts}.log"

header = (
    f"[run ] YOLO26-seg training (inline)\n"
    f"[cwd ] {PROJ}\n"
    f"[log ] {log_file}\n"
    f"[ts  ] {datetime.datetime.now().isoformat(timespec='seconds')}\n"
    f"[hyp ] imgsz=480 batch=4 workers=0 device='0' (còn lại: Ultralytics default)\n"
    f"{'-' * 70}\n"
)
print(header, end="")

log_fh = log_file.open("w", encoding="utf-8")
log_fh.write(header)

original_stdout = sys.stdout
sys.stdout = Tee(original_stdout, log_fh)

try:
    # ── Imports inside try so log captures everything ───────────────────────
    from ultralytics import YOLO

    # ── Load model (cached download from cell 1) ─────────────────────────────
    model = YOLO("yolo26n-seg.pt")
    print("[ok  ] model loaded.")

    # ── Train (Ultralytics prints a per-epoch table that streams live) ──────
    results = model.train(
        data     = str(PROJ / "data/processed/yolo_ecustfd_seg/ecustfd-seg.yaml"),
        imgsz    = 480,
        batch    = 4,
        workers  = 0,   # Windows: avoid multiprocessing pickling issues
        device   = "0", # GPU0 (RTX 4050 6 GB)
        project  = str(PROJ / "runs/yolo_seg"),
        name     = "ecustfd_yolo26seg",
        verbose  = True,
    )

    # ── Copy best/last checkpoint to models/ ────────────────────────────────
    save_dir = Path(results.save_dir)
    models_dir = PROJ / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    for tag in ("best", "last"):
        src = save_dir / "weights" / f"{tag}.pt"
        if src.exists():
            dst = models_dir / f"ecustfd_yolo26seg_{tag}.pt"
            shutil.copy2(src, dst)
            print(f"[ok  ] {src.name} -> {dst} ({dst.stat().st_size/1024/1024:.1f} MB)")

    print(f"\n[done] Training complete. Results: {save_dir}")
    print(f"[done] Log file: {log_file}")
finally:
    sys.stdout = original_stdout
    log_fh.close()

[run ] YOLO26-seg training (inline)
[cwd ] E:\AI_Research\dlt8
[log ] E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\logs\train_run_20260902_002456.log
[ts  ] 2026-09-02T00:24:56
[hyp ] imgsz=480 batch=4 workers=0 device='0' (còn lại: Ultralytics default)
----------------------------------------------------------------------
[ok  ] model loaded.
New https://pypi.org/project/ultralytics/8.4.137 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.87  Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\ecustfd-seg.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=

## Training Evaluation & Validation

Inspects training convergence via `results.csv`, logs configuration from `args.yaml`, and evaluates final mAP50 / mAP50-95 on the validation split. Run this cell after training completes.


In [4]:
# Cell 3 — Đánh giá kết quả train
#
# Tìm run directory mới nhất (Ultralytics tự thêm hậu tố -2, -3, ... nếu
# trùng tên), sau đó đọc:
#   - args.yaml       : snapshot hyper đã dùng (để audit)
#   - results.csv     : loss/mAP mỗi epoch (để kiểm tra hội tụ)
#   - weights/best.pt : checkpoint tốt nhất
from pathlib import Path
import pandas as pd
import yaml

PROJ = Path("E:/AI_Research/dlt8")
RUNS_DIR = PROJ / "runs" / "yolo_seg"

# Tìm thư mục run mới nhất
run_dirs = sorted(
    [d for d in RUNS_DIR.iterdir() if d.is_dir()],
    key=lambda d: d.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    raise RuntimeError(f"No run directory found under {RUNS_DIR}")

latest = run_dirs[0]
print(f"[latest run] {latest}\n")

# ── 1. In hyper đã dùng (snapshot từ args.yaml) ─────────────────────────────
args_yaml = latest / "args.yaml"
if args_yaml.exists():
    cfg = yaml.safe_load(args_yaml.read_text(encoding="utf-8"))
    print("─── Hyperparameters used (from args.yaml) ───")
    interesting = [
        "model", "data", "epochs", "batch", "imgsz", "device", "workers",
        "patience", "optimizer", "lr0", "lrf", "momentum", "weight_decay",
        "warmup_epochs", "warmup_momentum", "warmup_bias_lr",
        "close_mosaic", "amp", "cache", "freeze", "pretrained",
        "box", "cls", "dfl",
        "hsv_h", "hsv_s", "hsv_v",
        "degrees", "translate", "scale", "shear", "perspective",
        "flipud", "fliplr", "bgr",
        "mosaic", "mixup", "cutmix", "copy_paste", "erasing", "auto_augment",
    ]
    for k in interesting:
        if k in cfg:
            print(f"  {k:>22s} = {cfg[k]}")
    print()

# ── 2. Đọc results.csv và kiểm tra hội tụ ──────────────────────────────────
csv_path = latest / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    print(f"─── Training log ({len(df)} epochs) ───")
    print(f"  columns: {list(df.columns)}\n")

    # In 5 epoch đầu + 5 epoch cuối để xem đường cong
    n = len(df)
    print("First 5 epochs:")
    print(df.head().to_string(index=False))
    print("\nLast 5 epochs:")
    print(df.tail().to_string(index=False))

    # Các metric để đánh giá hội tụ
    print("\n─── Convergence check ───")
    metric_pairs = [
        ("train/seg_loss", "train box+seg loss"),
        ("val/seg_loss",   "val   box+seg loss"),
        ("metrics/mAP50(B)",       "val box mAP@0.5"),
        ("metrics/mAP50-95(B)",    "val box mAP@0.5:0.95"),
        ("metrics/mAP50(M)",       "val mask mAP@0.5"),
        ("metrics/mAP50-95(M)",    "val mask mAP@0.5:0.95"),
    ]
    for col, label in metric_pairs:
        if col in df.columns:
            first = float(df[col].iloc[0])
            best = float(df[col].max() if "mAP" in col else df[col].min())
            last = float(df[col].iloc[-1])
            best_epoch = int(df[col].idxmax() if "mAP" in col else df[col].idxmin()) + 1
            print(f"  {label:>26s}: epoch1={first:.4f}  best={best:.4f}@ep{best_epoch}  last={last:.4f}")

# ── 3. Kiểm tra checkpoint ─────────────────────────────────────────────────
weights_dir = latest / "weights"
best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"
print("\n─── Checkpoints ───")
if best_pt.exists():
    print(f"  best.pt  : {best_pt.stat().st_size / 1024 / 1024:.1f} MB  ({best_pt})")
if last_pt.exists():
    print(f"  last.pt  : {last_pt.stat().st_size / 1024 / 1024:.1f} MB  ({last_pt})")

models_dir = PROJ / "models"
print(f"\n─── models/ (production copies) ───")
for p in sorted(models_dir.glob("*_best.pt")) + sorted(models_dir.glob("*_last.pt")):
    print(f"  {p.name}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")

print(f"\n[done] Eval complete for {latest.name}")

[latest run] E:\AI_Research\dlt8\runs\yolo_seg\ecustfd_yolo26seg

─── Hyperparameters used (from args.yaml) ───
                   model = yolo26n-seg.pt
                    data = E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\ecustfd-seg.yaml
                  epochs = 100
                   batch = 4
                   imgsz = 480
                  device = 0
                 workers = 0
                patience = 100
               optimizer = auto
                     lr0 = 0.01
                     lrf = 0.01
                momentum = 0.937
            weight_decay = 0.0005
           warmup_epochs = 3.0
         warmup_momentum = 0.8
          warmup_bias_lr = 0.1
            close_mosaic = 10
                     amp = True
                   cache = False
                  freeze = None
              pretrained = True
                     box = 7.5
                     cls = 0.5
                     dfl = 1.5
                   hsv_h = 0.015
                   hsv_s = 0.